In [30]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())

import pandas as pd
import duckdb
from pathlib import Path
from config import RESEARCH_ROOT, NSE_DB_PATH

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

import numpy as np
from scipy import stats

In [31]:
print(os.getcwd())

e:\Projects


In [32]:
RESEARCH_DB_PATH = RESEARCH_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)

In [33]:
con.execute("""
    DESC daily_features
""").fetchall()

[('trade_date', 'DATE', 'YES', None, None, None),
 ('nifty_close', 'DOUBLE', 'YES', None, None, None),
 ('fwd_ret_1d', 'DOUBLE', 'YES', None, None, None),
 ('fwd_ret_5d', 'DOUBLE', 'YES', None, None, None),
 ('fwd_ret_20d', 'DOUBLE', 'YES', None, None, None),
 ('up_1d', 'INTEGER', 'YES', None, None, None),
 ('vix_close', 'DOUBLE', 'YES', None, None, None),
 ('pcr', 'DOUBLE', 'YES', None, None, None),
 ('max_pain_dist_pct', 'DOUBLE', 'YES', None, None, None),
 ('basis', 'DOUBLE', 'YES', None, None, None),
 ('cost_of_carry', 'DOUBLE', 'YES', None, None, None),
 ('fut_chng_oi_pct', 'DOUBLE', 'YES', None, None, None),
 ('underlying_daily_vol', 'DOUBLE', 'YES', None, None, None),
 ('futures_daily_vol', 'DOUBLE', 'YES', None, None, None),
 ('applicable_daily_vol', 'DOUBLE', 'YES', None, None, None),
 ('advances', 'INTEGER', 'YES', None, None, None),
 ('declines', 'INTEGER', 'YES', None, None, None),
 ('adv_decl_ratio', 'DOUBLE', 'YES', None, None, None),
 ('price_band_hits', 'INTEGER', 'YES'

In [34]:
con.execute("""
    SELECT MAX(trade_date), MIN(trade_date) FROM daily_features
""").fetchall()

[(datetime.date(2026, 7, 27), datetime.date(2016, 1, 1))]

In [35]:
# ── 1. Bootstrap CI for VIX_high + basis_mid cell ──────────────────────────
cell = con.execute("""
    SELECT fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE split = 'train'
      AND vix_close >= 18
      AND basis >= 50 AND basis < 100
      AND fwd_ret_5d IS NOT NULL
      AND fwd_ret_20d IS NOT NULL
""").df()

np.random.seed(42)
n_boot = 10000

boot_5d  = [np.mean(np.random.choice(cell['fwd_ret_5d'],  size=len(cell), replace=True)) for _ in range(n_boot)]
boot_20d = [np.mean(np.random.choice(cell['fwd_ret_20d'], size=len(cell), replace=True)) for _ in range(n_boot)]

print("=== Bootstrap CI — VIX_high + basis_mid (train) ===")
print(f"5D  return: mean={np.mean(boot_5d):.3f}%  95% CI [{np.percentile(boot_5d,2.5):.3f}, {np.percentile(boot_5d,97.5):.3f}]")
print(f"20D return: mean={np.mean(boot_20d):.3f}%  95% CI [{np.percentile(boot_20d,2.5):.3f}, {np.percentile(boot_20d,97.5):.3f}]")
print(f"P(5D > 0):  {np.mean(np.array(boot_5d) > 0):.3f}")
print(f"P(20D > 0): {np.mean(np.array(boot_20d) > 0):.3f}")

# ── 2. Regression with VIX × Basis interaction term ────────────────────────
from sklearn.linear_model import LinearRegression

df_train = con.execute("""
    SELECT 
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d,
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND cost_of_carry IS NOT NULL
      AND fut_chng_oi_pct IS NOT NULL
""").df()

# Standardize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

features = ['vix_close', 'basis', 'cost_of_carry', 'fut_chng_oi_pct']
X = df_train[features].copy()
X['vix_x_basis'] = df_train['vix_close'] * df_train['basis']  # interaction term
X_scaled = scaler.fit_transform(X)

# OLS with statsmodels for proper p-values
import statsmodels.api as sm
X_sm = sm.add_constant(X_scaled)
model = sm.OLS(df_train['fwd_ret_5d'], X_sm).fit()

print("\n=== OLS Regression — 5D Forward Return (train) ===")
coef_names = ['const'] + features + ['vix_x_basis']
for name, coef, pval in zip(coef_names, model.params, model.pvalues):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:20s}  coef={coef:+.4f}  p={pval:.4f}")

print(f"\nR²={model.rsquared:.4f}  Adj-R²={model.rsquared_adj:.4f}")

=== Bootstrap CI — VIX_high + basis_mid (train) ===
5D  return: mean=0.647%  95% CI [0.024, 1.313]
20D return: mean=1.224%  95% CI [0.085, 2.334]
P(5D > 0):  0.978
P(20D > 0): 0.982

=== OLS Regression — 5D Forward Return (train) ===
✅ const                 coef=+0.2605  p=0.0000
✅ vix_close             coef=+0.1302  p=0.0088
✅ basis                 coef=-0.8515  p=0.0000
✅ cost_of_carry         coef=-0.1252  p=0.0248
✅ fut_chng_oi_pct       coef=-0.1786  p=0.0005
✅ vix_x_basis           coef=+0.9856  p=0.0000

R²=0.0385  Adj-R²=0.0363


In [36]:
df_train2 = con.execute("""
    SELECT 
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND cost_of_carry IS NOT NULL
      AND fut_chng_oi_pct IS NOT NULL
""").df()

# Theoretically motivated transformations
df_train2['log_vix']      = np.log(df_train2['vix_close'])
df_train2['basis_sq']     = df_train2['basis'] ** 2
df_train2['log_vix_x_basis'] = df_train2['log_vix'] * df_train2['basis']

features_v2 = ['log_vix', 'basis', 'basis_sq', 'cost_of_carry', 
                'fut_chng_oi_pct', 'log_vix_x_basis']

X2 = df_train2[features_v2].copy()
X2_scaled = StandardScaler().fit_transform(X2)
X2_sm = sm.add_constant(X2_scaled)

model2 = sm.OLS(df_train2['fwd_ret_5d'], X2_sm).fit()

print("=== OLS v2 — Log VIX + Basis² + Interaction (train) ===")
coef_names2 = ['const'] + features_v2
for name, coef, pval in zip(coef_names2, model2.params, model2.pvalues):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}")

print(f"\nR²={model2.rsquared:.4f}  Adj-R²={model2.rsquared_adj:.4f}")

# Compare adjusted R² between models
print(f"\nModel 1 Adj-R²: 0.1971  (linear + VIX×basis)")
print(f"Model 2 Adj-R²: {model2.rsquared_adj:.4f}  (log VIX + basis² + interaction)")

=== OLS v2 — Log VIX + Basis² + Interaction (train) ===
✅ const                      coef=+0.2605  p=0.0000
   log_vix                    coef=+0.0618  p=0.2675
✅ basis                      coef=-2.6929  p=0.0000
   basis_sq                   coef=-0.0467  p=0.6684
   cost_of_carry              coef=-0.0688  p=0.2271
✅ fut_chng_oi_pct            coef=-0.1710  p=0.0010
✅ log_vix_x_basis            coef=+2.7627  p=0.0000

R²=0.0267  Adj-R²=0.0241

Model 1 Adj-R²: 0.1971  (linear + VIX×basis)
Model 2 Adj-R²: 0.0241  (log VIX + basis² + interaction)


In [37]:
PAPER_BLUE   = '#1f6f8b'
PAPER_ORANGE = '#d98c0f'
PAPER_RED    = '#b3293f'
PAPER_TEAL   = '#2a7f62'
PAPER_GOLD   = '#b8860b'
PAPER_GRAY   = '#888888'

GRID_COLOR   = '#e0e0e0'
AXIS_COLOR   = '#444444'
TEXT_COLOR   = '#1a1a1a'

In [38]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Publication style — reusable across all plots in the paper ──
PAPER_STYLE = {
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#444444',
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linewidth': 0.6,
    'grid.alpha': 0.7,
    'axes.labelcolor': '#1a1a1a',
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10.5,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'font.family': 'serif',
    'font.serif': ['Georgia', 'Times New Roman', 'DejaVu Serif'],
    'legend.frameon': True,
    'legend.facecolor': 'white',
    'legend.edgecolor': '#cccccc',
    'legend.fontsize': 9.5,
    'savefig.facecolor': 'white',
    'savefig.dpi': 300,
}
plt.rcParams.update(PAPER_STYLE)

# Muted, colorblind-conscious academic palette (replaces dark-theme neon accents)
PAPER_GOLD   = '#b8860b'
PAPER_GRAY   = '#888888'

# ── Data prep (unchanged) ──
df_plot = con.execute("""
    SELECT basis, fwd_ret_5d, ln(vix_close) as log_vix
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND basis IS NOT NULL
      AND vix_close IS NOT NULL
""").df()

p5, p95 = df_plot['basis'].quantile([0.05, 0.95])
df_plot['basis_w'] = df_plot['basis'].clip(p5, p95)

log_vix_median = df_plot['log_vix'].median()
basis_range = np.linspace(p5, p95, 200)

b1 = model2.params.iloc[2]
b2 = model2.params.iloc[3]
b3 = model2.params.iloc[4]

basis_mean = df_plot['basis_w'].mean()
basis_std  = df_plot['basis_w'].std()
basis_sq_mean = (df_plot['basis_w']**2).mean()
basis_sq_std  = (df_plot['basis_w']**2).std()
interaction_raw = df_plot['log_vix'] * df_plot['basis_w']
inter_mean = interaction_raw.mean()
inter_std  = interaction_raw.std()

basis_scaled    = (basis_range - basis_mean) / basis_std
basis_sq_scaled = (basis_range**2 - basis_sq_mean) / basis_sq_std
interaction_scaled = ((log_vix_median * basis_range) - inter_mean) / inter_std

predicted = (model2.params.iloc[0]
             + b1 * basis_scaled
             + b2 * basis_sq_scaled
             + b3 * interaction_scaled)

adj_r2 = model2.rsquared_adj
n_obs = len(df_plot)

# ── Figure ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Non-Linear Basis Effect on 5-Day Forward Returns',
             fontsize=13.5, fontweight='bold', color='#1a1a1a', y=1.02)

# Panel 1: quadratic fit
ax1.scatter(df_plot['basis'], df_plot['fwd_ret_5d'],
            alpha=0.35, color=PAPER_BLUE, s=14, edgecolors='none',
            label=f'Observed (n={n_obs})', zorder=2)
ax1.plot(basis_range, predicted, color=PAPER_ORANGE, linewidth=2.2,
         label='Conditional fit\n(at median VIX)', zorder=3)
ax1.axhline(0, color='#999999', linewidth=0.8, zorder=1)
ax1.axvline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=1)
ax1.set_xlabel('Basis (Futures − Spot)')
ax1.set_ylabel('5-Day Forward Return (%)')
ax1.set_title('(a) Quadratic Basis Effect', loc='left', fontsize=11)
ax1.legend(loc='upper right', fontsize=9)
ax1.text(0.02, 0.02, f'Adj. R² = {adj_r2:.3f}',
         transform=ax1.transAxes, fontsize=9, color='#555555',
         verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                    edgecolor='#cccccc', linewidth=0.6))

# Panel 2: VIX regime overlay
colors = {
    'Low VIX (<14)':   ('#1f6f8b', 'o'),   # strong teal-blue, circle
    'Mid VIX (14–18)': ('#d98c0f', '^'),   # strong amber, triangle
    'High VIX (≥18)':  ('#b3293f', 's'),   # strong red, square
}

for regime, (color, marker) in colors.items():
    if regime.startswith('Low'):
        mask = df_plot['log_vix'] < np.log(14)
    elif regime.startswith('Mid'):
        mask = (df_plot['log_vix'] >= np.log(14)) & (df_plot['log_vix'] < np.log(18))
    else:
        mask = df_plot['log_vix'] >= np.log(18)
    ax2.scatter(df_plot.loc[mask, 'basis'], df_plot.loc[mask, 'fwd_ret_5d'],
                alpha=0.75, color=color, s=32, marker=marker,
                edgecolors='white', linewidths=0.5,
                label=regime, zorder=3 if regime.startswith('High') else 2)

ax2.axhline(0, color='#999999', linewidth=0.8, zorder=1)
ax2.axvline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=1)
ax2.set_xlabel('Basis (Futures − Spot)')
ax2.set_ylabel('5-Day Forward Return (%)')
ax2.set_title('(b) Basis–Return Relationship by VIX Level', loc='left', fontsize=11)
ax2.legend(loc='upper right', fontsize=9, title='VIX Regime', title_fontsize=9.5)

for ax in (ax1, ax2):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(RESEARCH_ROOT / 'plots/basis_analysis.png', dpi=300, bbox_inches='tight')
print("saved")

saved


In [39]:
# Check if the basis² significance is driven by discount outliers
print("Basis distribution in train:")
print(df_plot['basis'].describe())
print(f"\nDays with basis < 0: {(df_plot['basis'] < 0).sum()}")
print(f"Days with basis > 150: {(df_plot['basis'] > 150).sum()}")

Basis distribution in train:
count    2346.000000
mean       27.141709
std        33.020446
min       -86.200000
25%         5.750000
50%        19.075000
75%        40.487500
max       225.600000
Name: basis, dtype: float64

Days with basis < 0: 366
Days with basis > 150: 16


In [40]:
from scipy.stats.mstats import winsorize

# Check 5th and 95th percentiles
p5  = df_plot['basis'].quantile(0.05)
p95 = df_plot['basis'].quantile(0.95)
print(f"Basis p5={p5:.1f}, p95={p95:.1f}")

# Rerun model with winsorized basis
df_train3 = con.execute("""
    SELECT 
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND cost_of_carry IS NOT NULL
      AND fut_chng_oi_pct IS NOT NULL
""").df()

df_train3['log_vix']       = np.log(df_train3['vix_close'])
df_train3['basis_w']       = df_train3['basis'].clip(lower=p5, upper=p95)
df_train3['basis_sq_w']    = df_train3['basis_w'] ** 2
df_train3['log_vix_x_basis_w'] = df_train3['log_vix'] * df_train3['basis_w']

features_v3 = ['log_vix', 'basis_w', 'basis_sq_w', 
                'cost_of_carry', 'fut_chng_oi_pct', 'log_vix_x_basis_w']

X3 = df_train3[features_v3].copy()
scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)
X3_sm = sm.add_constant(X3_scaled)

model3 = sm.OLS(df_train3['fwd_ret_5d'], X3_sm).fit()

print("\n=== OLS v3 — Winsorized Basis (train) ===")
coef_names3 = ['const'] + features_v3
for name, coef, pval in zip(coef_names3, model3.params, model3.pvalues):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}")

print(f"\nR²={model3.rsquared:.4f}  Adj-R²={model3.rsquared_adj:.4f}")
print(f"vs Model 1 Adj-R²: 0.1971")
print(f"vs Model 2 Adj-R²: 0.2185")

Basis p5=-10.4, p95=92.7

=== OLS v3 — Winsorized Basis (train) ===
✅ const                      coef=+0.2605  p=0.0000
   log_vix                    coef=+0.0468  p=0.4373
✅ basis_w                    coef=-1.9013  p=0.0012
   basis_sq_w                 coef=-0.0162  p=0.9155
   cost_of_carry              coef=-0.0232  p=0.6937
✅ fut_chng_oi_pct            coef=-0.1610  p=0.0025
✅ log_vix_x_basis_w          coef=+1.9036  p=0.0003

R²=0.0163  Adj-R²=0.0137
vs Model 1 Adj-R²: 0.1971
vs Model 2 Adj-R²: 0.2185


In [41]:
features_final = ['basis_w', 'basis_sq_w', 'log_vix_x_basis_w']

X_final = df_train3[features_final].copy()
scaler_final = StandardScaler()
X_final_scaled = scaler_final.fit_transform(X_final)
X_final_sm = sm.add_constant(X_final_scaled)

model_final = sm.OLS(df_train3['fwd_ret_5d'], X_final_sm).fit()

print("=== Final Parsimonious Model (train) ===")
for name, coef, pval, ci_low, ci_high in zip(
    ['const'] + features_final,
    model_final.params,
    model_final.pvalues,
    model_final.conf_int()[0],
    model_final.conf_int()[1]
):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}  95%CI=[{ci_low:+.4f}, {ci_high:+.4f}]")

print(f"\nAdj-R²={model_final.rsquared_adj:.4f}")
print(f"AIC={model_final.aic:.2f}")
print(f"BIC={model_final.bic:.2f}")

# Also run Durbin-Watson for autocorrelation (important for time series)
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(model_final.resid)
print(f"Durbin-Watson={dw:.4f}  (2.0=no autocorrelation, <2=positive autocorr)")

=== Final Parsimonious Model (train) ===
✅ const                      coef=+0.2605  p=0.0000  95%CI=[+0.1658, +0.3551]
✅ basis_w                    coef=-2.2631  p=0.0000  95%CI=[-3.2027, -1.3235]
   basis_sq_w                 coef=+0.0701  p=0.6130  95%CI=[-0.2016, +0.3418]
✅ log_vix_x_basis_w          coef=+2.1310  p=0.0000  95%CI=[+1.2738, +2.9882]

Adj-R²=0.0105
AIC=10023.58
BIC=10046.42
Durbin-Watson=0.3970  (2.0=no autocorrelation, <2=positive autocorr)


In [42]:
# HAC (Newey-West) standard errors — correct for autocorrelation and heteroskedasticity
# Standard in financial econometrics, expected by reviewers
model_hac = sm.OLS(df_train3['fwd_ret_5d'], X_final_sm).fit(
    cov_type='HAC', 
    cov_kwds={'maxlags': 5}  # 5 lags covers one trading week
)

print("=== Final Model with HAC Standard Errors (Newey-West, 5 lags) ===")
for name, coef, pval, ci_low, ci_high in zip(
    ['const'] + features_final,
    model_hac.params,
    model_hac.pvalues,
    model_hac.conf_int()[0],
    model_hac.conf_int()[1]
):
    sig = "✅" if pval < 0.05 else "❌"
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}  95%CI=[{ci_low:+.4f}, {ci_high:+.4f}]")

print(f"\nAdj-R²={model_hac.rsquared_adj:.4f}")

# Also check how bad the autocorrelation is at different lags
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(model_final.resid, lags=[1, 5, 10], return_df=True)
print("\n=== Ljung-Box Test (residual autocorrelation) ===")
print(lb_test)
print("(p < 0.05 means significant autocorrelation at that lag)")

=== Final Model with HAC Standard Errors (Newey-West, 5 lags) ===
✅ const                      coef=+0.2605  p=0.0049  95%CI=[+0.0789, +0.4420]
✅ basis_w                    coef=-2.2631  p=0.0040  95%CI=[-3.8056, -0.7206]
❌ basis_sq_w                 coef=+0.0701  p=0.7830  95%CI=[-0.4286, +0.5688]
✅ log_vix_x_basis_w          coef=+2.1310  p=0.0060  95%CI=[+0.6122, +3.6498]

Adj-R²=0.0105

=== Ljung-Box Test (residual autocorrelation) ===
        lb_stat  lb_pvalue
1   1432.470892        0.0
5   2708.956804        0.0
10  2712.808806        0.0
(p < 0.05 means significant autocorrelation at that lag)


In [43]:
# Plot residual diagnostics — paper style
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Assumes PAPER_STYLE rcParams + PAPER_BLUE/PAPER_ORANGE/PAPER_RED/PAPER_TEAL/PAPER_GOLD
# already applied earlier in the notebook (see basis_analysis.png setup).

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Residual Diagnostics', fontsize=13.5, fontweight='bold',
             color='#1a1a1a', y=1.02)

resid = model_final.resid
df_train3_sorted = df_train3.copy().reset_index(drop=True)
df_train3_sorted['resid'] = resid.values

# (a) Residual ACF
plot_acf(resid, lags=30, ax=axes[0, 0], color=PAPER_BLUE,
         zero=False, title="")
axes[0, 0].axhline(0, color='#999999', linewidth=0.8)
axes[0, 0].set_title('(a) Residual Autocorrelation Function', loc='left', fontsize=11)
axes[0, 0].set_xlabel('Lag')
axes[0, 0].set_ylabel('ACF')

# (b) Normal Q-Q Plot
stats.probplot(resid, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('')
axes[0, 1].get_lines()[0].set_markerfacecolor(PAPER_BLUE)
axes[0, 1].get_lines()[0].set_markeredgecolor(PAPER_BLUE)
axes[0, 1].get_lines()[0].set_alpha(0.55)
axes[0, 1].get_lines()[0].set_markersize(4)
axes[0, 1].get_lines()[1].set_color(PAPER_ORANGE)
axes[0, 1].get_lines()[1].set_linewidth(2.2)
axes[0, 1].set_title('(b) Normal Q-Q Plot', loc='left', fontsize=11)
axes[0, 1].set_xlabel('Theoretical Quantiles')
axes[0, 1].set_ylabel('Sample Quantiles')

# (c) Residuals vs VIX + LOWESS trend
axes[1, 0].scatter(df_train3_sorted['vix_close'], df_train3_sorted['resid'],
                    color=PAPER_BLUE, alpha=0.45, s=18, edgecolors='none', zorder=2)
smooth = lowess(df_train3_sorted['resid'], df_train3_sorted['vix_close'], frac=0.3)
axes[1, 0].plot(smooth[:, 0], smooth[:, 1], color=PAPER_ORANGE,
                 linewidth=2.2, label='LOWESS trend', zorder=3)
axes[1, 0].axhline(0, color='#999999', linewidth=0.8, zorder=1)
axes[1, 0].set_title('(c) Residuals vs VIX', loc='left', fontsize=11)
axes[1, 0].set_xlabel('VIX')
axes[1, 0].set_ylabel('Residual')
axes[1, 0].legend(loc='upper right', fontsize=9, frameon=True,
                   facecolor='white', edgecolor='#cccccc')

# (d) Rolling 20-Day Residual Mean
rolling_resid = pd.Series(resid.values).rolling(20).mean()
x_idx = np.arange(len(rolling_resid))
axes[1, 1].plot(x_idx, rolling_resid, color=PAPER_GOLD, linewidth=1.8, zorder=3)
axes[1, 1].fill_between(x_idx, rolling_resid, 0,
                         where=rolling_resid > 0, alpha=0.3, color=PAPER_TEAL, zorder=2)
axes[1, 1].fill_between(x_idx, rolling_resid, 0,
                         where=rolling_resid < 0, alpha=0.3, color=PAPER_RED, zorder=2)
axes[1, 1].axhline(0, color='#999999', linewidth=0.8, zorder=1)
axes[1, 1].set_title('(d) Rolling 20-Day Residual Mean', loc='left', fontsize=11)
axes[1, 1].set_xlabel('Trading Day')
axes[1, 1].set_ylabel('Residual (20D rolling mean)')

for ax in axes.flatten():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(RESEARCH_ROOT / 'plots/residual_diagnostics.png',
            dpi=300, bbox_inches='tight', facecolor='white')
print("saved")

saved


In [44]:
import subprocess
subprocess.run(['pip', 'install', 'hmmlearn', '--break-system-packages', '-q'])

from hmmlearn import hmm

# Pull training data for HMM
df_hmm = con.execute("""
    SELECT
        trade_date,
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d
    FROM daily_features
    WHERE split = 'train'
    AND fwd_ret_5d IS NOT NULL
    AND vix_close IS NOT NULL
    AND basis IS NOT NULL
    AND cost_of_carry IS NOT NULL
    AND fut_chng_oi_pct IS NOT NULL
    ORDER BY trade_date
""").df()

print(f"HMM training observations: {len(df_hmm)}")
print(df_hmm.describe().round(3))

HMM training observations: 2233
                       trade_date  vix_close     basis  cost_of_carry  \
count                        2233   2233.000  2233.000       2233.000   
mean   2020-09-27 02:51:32.163009     16.844    28.571          0.055   
min           2016-01-01 00:00:00     10.135   -86.200         -0.496   
25%           2018-05-16 00:00:00     13.378     7.550          0.019   
50%           2020-09-29 00:00:00     15.392    20.450          0.047   
75%           2023-02-07 00:00:00     18.488    42.200          0.077   
max           2025-06-20 00:00:00     83.608   225.600          1.946   
std                           NaN      6.381    33.161          0.104   

       fut_chng_oi_pct  fwd_ret_5d  
count         2233.000    2233.000  
mean            -2.406       0.260  
min            -41.504     -19.024  
25%             -3.850      -0.913  
50%             -0.740       0.420  
75%              1.743       1.558  
max             22.330      10.522  
std           

In [45]:
print("df_hmm shape:", df_hmm.shape)

# Check nulls in source
check = con.execute("""
    SELECT 
        COUNT(*) as total,
        COUNT(vix_close) as has_vix,
        COUNT(basis) as has_basis,
        COUNT(fut_chng_oi_pct) as has_oi,
    FROM daily_features
    WHERE split = 'train'
""").df()
print(check)

# Find the missing dates
all_dates = con.execute("""
    SELECT trade_date FROM daily_features 
    WHERE split = 'train' ORDER BY trade_date
""").df()

print(f"\nTotal train rows: {len(all_dates)}")
print(f"After null filter: {len(df_hmm)}")
print(f"Missing: {len(all_dates) - len(df_hmm)}")

# Which dates are missing
missing = all_dates[~all_dates['trade_date'].isin(df_hmm['trade_date'])]
print("\nMissing dates:")
print(missing)

df_hmm shape: (2233, 6)
   total  has_vix  has_basis  has_oi
0   2346     2346       2346    2346

Total train rows: 2346
After null filter: 2233
Missing: 113

Missing dates:
     trade_date
18   2016-01-28
38   2016-02-25
60   2016-03-31
77   2016-04-28
97   2016-05-26
...         ...
2250 2025-01-30
2270 2025-02-27
2289 2025-03-27
2305 2025-04-24
2329 2025-05-29

[113 rows x 1 columns]


In [46]:
df_hmm_full = con.execute("""
    SELECT trade_date, vix_close, basis, fut_chng_oi_pct
    FROM daily_features
    WHERE split = 'train'
    ORDER BY trade_date
""").df()

# Forward fill expiry nulls only
df_hmm_full['fut_chng_oi_pct'] = df_hmm_full['fut_chng_oi_pct'].ffill()
df_hmm_full['basis'] = df_hmm_full['basis'].ffill()

# Verify
print("Nulls remaining (fut_chng_oi_pct):", df_hmm_full['fut_chng_oi_pct'].isna().sum())
print("Nulls remaining (basis):", df_hmm_full['basis'].isna().sum())
print("Shape:", df_hmm_full.shape)  # should be (249, 4)

df_hmm_full['log_vix'] = np.log(df_hmm_full['vix_close'])
df_hmm_full['basis_w'] = df_hmm_full['basis'].clip(
    df_hmm_full['basis'].quantile(0.05),
    df_hmm_full['basis'].quantile(0.95)
)

features_hmm = ['log_vix', 'basis_w', 'fut_chng_oi_pct']
scaler_hmm = StandardScaler()
X_scaled = scaler_hmm.fit_transform(df_hmm_full[features_hmm])

print("X_scaled shape:", X_scaled.shape)  # must be (249, 3)
print("means:", X_scaled.mean(axis=0).round(10))
print("stds: ", X_scaled.std(axis=0).round(4))

Nulls remaining (fut_chng_oi_pct): 0
Nulls remaining (basis): 0
Shape: (2346, 4)
X_scaled shape: (2346, 3)
means: [-0. -0. -0.]
stds:  [1. 1. 1.]


In [47]:
from joblib import Parallel, delayed
import numpy as np
from hmmlearn import hmm

def fit_one_seed(X_scaled, n, cov_type, seed):
    """Fit a single HMM seed; returns (logL, model, states) or None if invalid."""
    try:
        model = hmm.GaussianHMM(
            n_components=n,
            covariance_type=cov_type,
            n_iter=500,
            random_state=seed,
            tol=1e-5,
            init_params='stmc'
        )
        model.fit(X_scaled)
        logL = model.score(X_scaled)

        if np.any(model.transmat_.max(axis=1) > 0.99):
            return None
        _, states = model.decode(X_scaled)
        if np.any(np.bincount(states, minlength=n) < 0.10 * len(X_scaled)):
            return None
        return (logL, model, states)
    except Exception:
        return None


print("=== Canonical HMM State Selection (parallel) ===")

all_results = []
N_JOBS = -1   # all cores; drop to e.g. 6 if you want to keep the machine usable meanwhile

for cov_type in ['diag', 'full']:
    print(f"\n-- covariance_type='{cov_type}' --")

    for n in range(2, 9):
        seed_results = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(fit_one_seed)(X_scaled, n, cov_type, seed)
            for seed in range(50)
        )
        seed_results = [r for r in seed_results if r is not None]

        if not seed_results:
            print(f"  n={n}  no valid solution found")
            continue

        best_logL, best_model, best_states = max(seed_results, key=lambda r: r[0])

        d = len(features_hmm)
        cov_params = n * d * (d + 1) // 2 if cov_type == 'full' else n * d
        n_params = n * (n - 1) + n * d + cov_params
        bic = -2 * best_logL + n_params * np.log(len(X_scaled))

        occupancy = (np.bincount(best_states, minlength=n) / len(X_scaled) * 100).round(1)

        print(f"  n={n}  logL={best_logL:.2f}  n_params={n_params:3d}"
              f"  BIC={bic:.2f}  occupancy={occupancy}")

        all_results.append({
            'cov': cov_type, 'n': n, 'bic': bic,
            'logL': best_logL, 'model': best_model,
            'states': best_states
        })

valid = [r for r in all_results if r['model'] is not None]
best = min(valid, key=lambda x: x['bic'])
print(f"\n=== FINAL: cov={best['cov']}, n={best['n']}, BIC={best['bic']:.2f} ===")

=== Canonical HMM State Selection (parallel) ===

-- covariance_type='diag' --
  n=2  logL=-7974.48  n_params= 14  BIC=16057.60  occupancy=[35.5 64.5]
  n=3  logL=-7203.48  n_params= 24  BIC=14593.20  occupancy=[23.  32.8 44.2]
  n=4  logL=-6586.48  n_params= 36  BIC=13452.33  occupancy=[21.  36.7 24.6 17.6]
  n=5  logL=-6178.09  n_params= 50  BIC=12744.20  occupancy=[25.4 19.1 22.8 16.4 16.3]
  n=6  no valid solution found
  n=7  no valid solution found
  n=8  no valid solution found

-- covariance_type='full' --
  n=2  logL=-7903.64  n_params= 20  BIC=15962.50  occupancy=[35.6 64.4]
  n=3  logL=-7109.68  n_params= 33  BIC=14475.45  occupancy=[43.7 23.4 32.8]
  n=4  logL=-6507.85  n_params= 48  BIC=13388.20  occupancy=[33.9 27.2 17.  22. ]
  n=5  logL=-6143.94  n_params= 65  BIC=12792.31  occupancy=[17.8 25.6 24.3 16.5 15.9]
  n=6  logL=-5747.56  n_params= 84  BIC=12146.99  occupancy=[13.8 16.8 16.8 19.9 21.4 11.3]
  n=7  no valid solution found
  n=8  no valid solution found

=== FIN

In [48]:
import pickle

# Sanity check before saving — confirm this really is the locked canonical model
print(f"cov={best['cov']}, n={best['n']}, BIC={best['bic']:.2f}")

(RESEARCH_ROOT / 'models').mkdir(exist_ok=True)

with open(RESEARCH_ROOT / 'models/hmm_canonical.pkl', 'wb') as f:
    pickle.dump({
        'hmm_model': best['model'],
        'scaler': scaler_hmm,
        'basis_p5': p5,
        'basis_p95': p95,
        'features_hmm': features_hmm,   # ['log_vix', 'basis_w', 'fut_chng_oi_pct'] — order matters
    }, f)

print("Saved canonical HMM model + scaler + winsorization bounds")

cov=full, n=6, BIC=12146.99
Saved canonical HMM model + scaler + winsorization bounds


In [49]:
best5d = next(r for r in valid if r['cov'] == 'diag' and r['n'] == 5)
m5 = best5d['model']
s5 = best5d['states']

n_fit = X_scaled.shape[0]
n_states_total = sum((s5 == i).sum() for i in range(5))
assert n_states_total == n_fit, (
    f"State count {n_states_total} != fit sample size {n_fit} — "
    f"you're looking at a stale model/states object, rerun from cell 14"
)

print("Transition matrix:")
print(np.round(m5.transmat_, 3))
print("\nMeans (log_vix, basis_w, fut_chng_oi_pct):")
for i, mean in enumerate(m5.means_):
    print(f"  S{i}: {np.round(mean, 3)}  n={( s5==i).sum()}")
print("\nSelf-transition probs:")
for i in range(5):
    print(f"  S{i}: {m5.transmat_[i,i]:.3f}")

Transition matrix:
[[0.907 0.    0.014 0.003 0.077]
 [0.035 0.898 0.043 0.009 0.015]
 [0.008 0.    0.909 0.    0.083]
 [0.009 0.    0.    0.933 0.057]
 [0.077 0.115 0.055 0.049 0.704]]

Means (log_vix, basis_w, fut_chng_oi_pct):
  S0: [ 0.134 -0.14   0.343]  n=597
  S1: [-0.49   1.642  0.389]  n=447
  S2: [-0.807 -0.307  0.385]  n=536
  S3: [ 1.572 -0.506  0.284]  n=384
  S4: [-0.06  -0.747 -1.731]  n=382

Self-transition probs:
  S0: 0.907
  S1: 0.898
  S2: 0.909
  S3: 0.933
  S4: 0.704


In [50]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Attach states to df_hmm_full ─────────────────────────────────────────────
df_hmm_full['state'] = best['states']
N_STATES = best['n']

# ── Save to research.db ──────────────────────────────────────────────────────
con.execute("""
    ALTER TABLE daily_features
    ADD COLUMN IF NOT EXISTS hmm_state INTEGER;
""")

for _, row in df_hmm_full.iterrows():
    con.execute("""
        UPDATE daily_features
        SET hmm_state = ?
        WHERE trade_date = ?
          AND split = 'train'
    """, [int(row['state']), row['trade_date']])

print("States saved to daily_features")

results = con.execute("""
    SELECT
        hmm_state,
        COUNT(*) AS days,
        ROUND(AVG(fwd_ret_5d), 3) AS avg_5d,
        ROUND(
            AVG(
                CASE WHEN fwd_ret_5d > 0
                     THEN 1.0
                     ELSE 0.0
                END
            ) * 100,
            1
        ) AS pct_up
    FROM daily_features
    WHERE split = 'train'
      AND hmm_state IS NOT NULL
    GROUP BY hmm_state
    ORDER BY hmm_state
""").fetchall()

for row in results:
    print(row)

# ── Paper regime palette ─────────────────────────────────────────────────────
PAPER_PURPLE = '#7a6aa6'  # muted lavender — extension to the core 5-color
                          # palette, used only for this 5-state HMM figure

import matplotlib.cm as cm
_palette_base = [PAPER_BLUE, PAPER_RED, PAPER_ORANGE, PAPER_TEAL, PAPER_GOLD, PAPER_GRAY, PAPER_PURPLE]
if N_STATES <= len(_palette_base):
    state_color_list = _palette_base[:N_STATES]
else:
    state_color_list = [cm.tab10(i / N_STATES) for i in range(N_STATES)]
STATE_COLORS = {i: state_color_list[i] for i in range(N_STATES)}
STATE_LABELS = {i: f'S{i}' for i in range(N_STATES)} 

# Neutral foreground line color — used for ALL data series drawn over
# regime-shaded backgrounds, so color is reserved exclusively for regime
# identity and never collides with it.
REGIME_LINE_COLOR = '#444444'

# ── Figure Layout ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    4,
    1,
    figsize=(16, 10),
    gridspec_kw={'height_ratios': [1.2, 2, 1.3, 1.3]}
)

fig.suptitle(
    'Hidden Markov Model Regimes and Market Dynamics',
    fontsize=15,
    fontweight='bold',
    color='#1a1a1a',
    y=0.995
)

dates = df_hmm_full['trade_date'].values
states = df_hmm_full['state'].values

# ── Common Formatting ────────────────────────────────────────────────────────
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ============================================================================
# (a) Hidden State Timeline
# ============================================================================
ax = axes[0]

ax.step(
    dates,
    states,
    where='post',
    color='#999999',
    linewidth=1.2,
    zorder=1
)

ax.scatter(
    dates,
    states,
    c=[STATE_COLORS[s] for s in states],
    s=14,
    zorder=2
)

ax.set_yticks(list(range(N_STATES)))
ax.set_yticklabels([f'S{i}' for i in range(N_STATES)])

ax.set_ylabel('State')
ax.set_title(
    '(a) Inferred Hidden Market Regimes',
    loc='left',
    fontsize=11
)

patches = [mpatches.Patch(color=STATE_COLORS[s], label=STATE_LABELS[s]) for s in range(N_STATES)]

ax.legend(
    handles=patches,
    bbox_to_anchor=(1.01, 1.0),
    loc='upper left',
    fontsize=8,
    frameon=True,
    facecolor='white',
    edgecolor='#cccccc',
    borderaxespad=0
)

# ============================================================================
# (b) VIX with Regime Overlay
# ============================================================================
ax = axes[1]

for i in range(len(dates) - 1):
    ax.axvspan(
        dates[i],
        dates[i + 1],
        color=STATE_COLORS[states[i]],
        alpha=0.35,
        linewidth=0
    )

ax.plot(
    dates,
    df_hmm_full['vix_close'],
    color=REGIME_LINE_COLOR,
    linewidth=1.2,
    zorder=3
)

ax.set_ylabel('VIX')
ax.set_title(
    '(b) VIX Index Across Hidden Regimes',
    loc='left',
    fontsize=11
)

# ============================================================================
# (c) Futures Basis
# ============================================================================
ax = axes[2]

for i in range(len(dates) - 1):
    ax.axvspan(
        dates[i],
        dates[i + 1],
        color=STATE_COLORS[states[i]],
        alpha=0.35,
        linewidth=0
    )

ax.plot(
    dates,
    df_hmm_full['basis'],
    color=REGIME_LINE_COLOR,
    linewidth=1.2,
    zorder=3
)

ax.axhline(
    0,
    color='#999999',
    linewidth=0.8
)

ax.set_ylabel('Basis')
ax.set_title(
    '(c) Futures Basis by Regime',
    loc='left',
    fontsize=11
)

# ============================================================================
# (d) Open Interest Change
# ============================================================================
ax = axes[3]

for i in range(len(dates) - 1):
    ax.axvspan(
        dates[i],
        dates[i + 1],
        color=STATE_COLORS[states[i]],
        alpha=0.35,
        linewidth=0
    )

ax.plot(
    dates,
    df_hmm_full['fut_chng_oi_pct'],
    color=REGIME_LINE_COLOR,
    linewidth=1.2,
    zorder=3
)

ax.axhline(
    0,
    color='#999999',
    linewidth=0.8
)

ax.set_ylabel('OI Change (%)')
ax.set_title(
    '(d) Futures Open Interest Change by Regime',
    loc='left',
    fontsize=11
)

# ── Layout ───────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0, 0, 0.92, 0.97])

plt.savefig(
    RESEARCH_ROOT / 'plots/hmm_regimes.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved: hmm_regimes.png")

States saved to daily_features
(0, 323, 0.565, 64.1)
(1, 395, 0.249, 57.0)
(2, 394, 0.029, 53.6)
(3, 466, 0.277, 60.1)
(4, 502, 0.116, 59.8)
(5, 266, 0.654, 59.0)
Saved: hmm_regimes.png


C:\Users\sriva\AppData\Local\Temp\ipykernel_11408\1857183066.py:244: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [51]:
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# Pull train data with hmm_state
df = con.execute("""
    SELECT trade_date, hmm_state,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct, fii_pe_net_pct,
           market_cap_cr,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE split = 'train' AND hmm_state IS NOT NULL
    ORDER BY trade_date
""").df()

# market_cap_cr detrended — raw level is a time-trend proxy, use rolling z-score instead
roll_win = 252  # ~1 trading year
df['market_cap_z'] = (
    (df['market_cap_cr'] - df['market_cap_cr'].rolling(roll_win, min_periods=60).mean())
    / df['market_cap_cr'].rolling(roll_win, min_periods=60).std()
)

# max_pain_dist_pct — the linear IC is near-zero but the effect only shows up
# once you're away from max pain; use signed-magnitude so the composite can
# still pick up direction while the |x|>1.5% regime effect dominates
df['max_pain_dist_abs'] = df['max_pain_dist_pct'].abs()

# VIX x basis interaction — significant in the OLS work, not yet in the composite
df['vix_x_basis'] = df['vix_close'] * df['basis']

# client_fut_net_pct — contrarian effect is concentrated in the "heavy long"
# bucket (20-50), not linear; a squared/deviation-from-neutral transform
# captures "how far from neutral" without assuming a sign-consistent slope
df['client_fut_net_dev'] = (df['client_fut_net_pct'] - df['client_fut_net_pct'].median()).abs()

state_labels = {
    **{i: f'S{i}\n(n={(df["hmm_state"] == i).sum()})' for i in range(N_STATES)},
    'ALL': f'Full Sample\n(n={len(df)})',
}

factors = {
    'VIX':          'vix_close',
    'PCR':          'pcr',
    'MaxPainDist':  'max_pain_dist_pct',
    'MaxPainAbs':   'max_pain_dist_abs',      # new
    'Basis':        'basis',
    'CostOfCarry':  'cost_of_carry',
    'FutOIChng':    'fut_chng_oi_pct',
    'FII_Fut':      'fii_fut_net_pct',
    'Client_Fut':   'client_fut_net_pct',
    'ClientFutDev': 'client_fut_net_dev',      # new
    'FII_PE':       'fii_pe_net_pct',          # new
    'MCapZ':        'market_cap_z',            # new
    'VIXxBasis':    'vix_x_basis',             # new
}
horizons = {'1D': 'fwd_ret_1d', '5D': 'fwd_ret_5d', '20D': 'fwd_ret_20d'}
state_names = {i: f'S{i}' for i in range(N_STATES)}

def spearman_ic(x, y):
    mask = x.notna() & y.notna()
    if mask.sum() < 10:
        return np.nan, np.nan
    ic, p = stats.spearmanr(x[mask], y[mask])
    return ic, p

def sig_star(p):
    if np.isnan(p):
        return ''
    if p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.10:
        return '†'
    return ''

# separators now at every 3 rows for 6 groups
separator_rows = list(range(3, N_STATES * 3, 3))

# Build IC table
results = {}
for state_key, state_label in state_labels.items():
    for hz_name, hz_col in horizons.items():
        row_key = (state_label, hz_name)
        results[row_key] = {}
        sub = df if state_key == 'ALL' else df[df['hmm_state'] == state_key]
        for fac_name, fac_col in factors.items():
            results[row_key][fac_name] = spearman_ic(sub[fac_col], sub[hz_col])

row_order = [(sl, hz) for sl in state_labels.values() for hz in horizons]
col_order  = list(factors.keys())

ic_mat = np.full((len(row_order), len(col_order)), np.nan)
p_mat  = np.full((len(row_order), len(col_order)), np.nan)
for i, rk in enumerate(row_order):
    for j, fac in enumerate(col_order):
        ic_mat[i, j], p_mat[i, j] = results[rk][fac]

# ── Paper diverging palette: built from the locked red/blue, no green ──────
PAPER_DIVERGING = LinearSegmentedColormap.from_list(
    'paper_diverging', [PAPER_RED, '#ffffff', PAPER_BLUE]
)

fig, ax = plt.subplots(figsize=(14, 12))

vmax = 0.55
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(ic_mat, cmap=PAPER_DIVERGING, norm=norm, aspect='auto')

for i in range(len(row_order)):
    for j in range(len(col_order)):
        ic_val, p_val = ic_mat[i, j], p_mat[i, j]
        if np.isnan(ic_val):
            txt, color = 'n/a', PAPER_GRAY
        else:
            txt = f'{ic_val:+.2f}{sig_star(p_val)}'
            color = 'white' if abs(ic_val) > 0.25 else '#1a1a1a'
        ax.text(j, i, txt, ha='center', va='center',
                fontsize=13, color=color, fontweight='bold')

ax.set_xticks(range(len(col_order)))
ax.set_xticklabels(col_order, color='#1a1a1a', fontsize=11, ha='center')
ax.set_yticks(range(len(row_order)))
ax.set_yticklabels([f'{sl.split(chr(10))[0]}  {hz}' for sl, hz in row_order],
                   color='#1a1a1a', fontsize=11)

# Thin white cell borders (minor-tick grid trick) — major grid turned off
# so the global PAPER_STYLE gridlines don't cut through cell centers.
ax.grid(which='major', visible=False)
ax.set_xticks(np.arange(-0.5, len(col_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=0.6)
ax.tick_params(which='minor', bottom=False, left=False)

# Group separators — heavier than the cell grid, distinct purpose
for i in separator_rows:
    ax.axhline(i - 0.5, color='#444444', linewidth=1.2)

cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
cbar.ax.yaxis.set_tick_params(color='#444444')
cbar.ax.set_ylabel('Spearman IC', color='#1a1a1a', fontsize=11)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#1a1a1a', fontsize=11)
cbar.outline.set_edgecolor('#cccccc')

ax.tick_params(colors='#444444', which='major')
for spine in ax.spines.values():
    spine.set_edgecolor('#444444')
    spine.set_linewidth(0.8)

ax.set_title('Regime-Conditioned Factor IC — HMM States vs Full Sample\n'
             '** p<0.01  * p<0.05  † p<0.10  |  Train Set Only',
             loc='center', color='#1a1a1a', fontsize=15, pad=14)

plt.tight_layout()
plt.savefig(RESEARCH_ROOT / 'plots/hmm_regime_ic_heatmap.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")

done


C:\Users\sriva\AppData\Local\Temp\ipykernel_11408\2260345322.py:158: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [52]:
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Compute full-sample and per-state ICs
records = []
for hz_name, hz_col in horizons.items():
    for fac_name, fac_col in factors.items():
        ic_full, p_full = spearman_ic(df[fac_col], df[hz_col])
        records.append({
            'factor': fac_name, 'horizon': hz_name,
            'state': 'Full', 'ic': ic_full, 'p': p_full
        })
        for state_id, state_label in state_names.items():
            sub = df[df['hmm_state'] == state_id]
            ic, p = spearman_ic(sub[fac_col], sub[hz_col])
            records.append({
                'factor': fac_name, 'horizon': hz_name,
                'state': state_label, 'ic': ic, 'p': p
            })

res = pd.DataFrame(records)

def best_state_row(group):
    states_only = group[group['state'] != 'Full'].dropna(subset=['ic'])
    if states_only.empty:
        return None
    return states_only.loc[states_only['ic'].abs().idxmax()]

summary = []
for (fac, hz), grp in res.groupby(['factor', 'horizon']):
    full_row  = grp[grp['state'] == 'Full'].iloc[0]
    best_row  = best_state_row(grp)
    if best_row is None:
        continue
    summary.append({
        'factor':     fac,
        'horizon':    hz,
        'ic_full':    full_row['ic'],
        'p_full':     full_row['p'],
        'ic_best':    best_row['ic'],
        'p_best':     best_row['p'],
        'best_state': best_row['state'],
        'abs_gain':   abs(best_row['ic']) - abs(full_row['ic']),
    })

sdf = pd.DataFrame(summary)

hz_order  = ['1D', '5D', '20D']
fac_order = list(factors.keys())
BAR_W = 0.35

state_colors = {state_names[i]: STATE_COLORS[i] for i in range(N_STATES)}

def star(p):
    if np.isnan(p): return ''
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '†'
    return ''

panel_letters = {'1D': '(a)', '5D': '(b)', '20D': '(c)'}

fig, axes = plt.subplots(3, 1, figsize=(14, 13))
fig.suptitle(
    'Full-Sample IC vs Best Regime IC — Factor Predictability Lift\n'
    '** p<0.01  * p<0.05  † p<0.10  |  Train Set Only',
    fontsize=18, fontweight='bold', color='#1a1a1a', y=0.99
)

x = np.arange(len(fac_order))

for ax, hz in zip(axes, hz_order):
    sub = sdf[sdf['horizon'] == hz].set_index('factor').reindex(fac_order)

    ic_full = sub['ic_full'].values
    ic_best = sub['ic_best'].values
    p_full  = sub['p_full'].values
    p_best  = sub['p_best'].values
    best_states = sub['best_state'].values

    ax.bar(x - BAR_W/2, ic_full, BAR_W,
           color=PAPER_GRAY, alpha=0.75, label='Full Sample', zorder=3)

    for i, (val, bs) in enumerate(zip(ic_best, best_states)):
        color = state_colors.get(bs, PAPER_GRAY) if isinstance(bs, str) else PAPER_GRAY
        ax.bar(x[i] + BAR_W/2, val, BAR_W, color=color, alpha=0.85, zorder=3)

    # Significance annotations above bars — now bold for contrast
    for i, (vf, pf, vb, pb) in enumerate(zip(ic_full, p_full, ic_best, p_best)):
        sf, sb = star(pf), star(pb)
        offset = 0.01
        if not np.isnan(vf) and sf:
            ax.text(x[i] - BAR_W/2, vf + np.sign(vf)*offset, sf,
                    ha='center', va='bottom' if vf >= 0 else 'top',
                    color='#000000', fontsize=13, fontweight='bold')
        if not np.isnan(vb) and sb:
            ax.text(x[i] + BAR_W/2, vb + np.sign(vb)*offset, sb,
                    ha='center', va='bottom' if vb >= 0 else 'top',
                    color='#000000', fontsize=13, fontweight='bold')

    # Best-state label — anchored at the zero line, offset in points
    # (not data units) so the glyph fully clears axhline(0) regardless
    # of how each panel's y-range happens to be scaled
    for i, (bs, val) in enumerate(zip(best_states, ic_best)):
        if isinstance(bs, str):
            short = bs.split(':')[0]  # "S0", "S1" etc
            is_pos = (not np.isnan(val)) and val >= 0
            ax.annotate(
                short,
                xy=(x[i] + BAR_W / 2, 0),
                xytext=(0, 4 if is_pos else -4),
                textcoords='offset points',
                ha='center',
                va='bottom' if is_pos else 'top',
                fontsize=12, color='#1a1a1a', fontweight='bold', zorder=4
            )

    ax.axhline(0, color='#999999', linewidth=0.8, zorder=2)

    # Lift arrows for gains > 0.10
    for i, gain in enumerate(sub['abs_gain'].values):
        if not np.isnan(gain) and gain > 0.10:
            vf = ic_full[i] if not np.isnan(ic_full[i]) else 0
            vb = ic_best[i] if not np.isnan(ic_best[i]) else 0
            ax.annotate('', xy=(x[i] + BAR_W/2, vb),
                        xytext=(x[i] - BAR_W/2, vf),
                        arrowprops=dict(arrowstyle='->', color='#444444',
                                        lw=1.0, connectionstyle='arc3,rad=0.25'),
                        zorder=5)

    ax.set_xlim(-0.6, len(fac_order) - 0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(fac_order if hz == '20D' else [''] * len(fac_order),
                       fontsize=13)
    ax.set_ylabel(f'IC ({hz})', fontsize=13)
    ax.set_title(f'{panel_letters[hz]} {hz} Horizon', loc='left', fontsize=13)
    ax.tick_params(axis='y', labelsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, color='#e0e0e0', linewidth=0.6, zorder=1)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)

    if hz == '20D':
        patches = [mpatches.Patch(color=PAPER_GRAY, alpha=0.75, label='Full Sample')]
        for sname, scol in state_colors.items():
            patches.append(mpatches.Patch(color=scol, alpha=0.85, label=sname))
        ax.legend(handles=patches, loc='upper right', fontsize=11,
                  frameon=True, facecolor='white', edgecolor='#cccccc')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(RESEARCH_ROOT / 'plots/regime_ic_lift.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")

done


C:\Users\sriva\AppData\Local\Temp\ipykernel_11408\1076227734.py:156: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [53]:
# Pull train data with hmm_state
df = con.execute("""
    SELECT trade_date, hmm_state,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct, fii_pe_net_pct,
           market_cap_cr,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d,
           split,
    FROM daily_features
    WHERE hmm_state IS NOT NULL
    ORDER BY trade_date
""").df()

# market_cap_cr detrended — raw level is a time-trend proxy, use rolling z-score instead
roll_win = 252  # ~1 trading year
df['market_cap_z'] = (
    (df['market_cap_cr'] - df['market_cap_cr'].rolling(roll_win, min_periods=60).mean())
    / df['market_cap_cr'].rolling(roll_win, min_periods=60).std()
)

# max_pain_dist_pct — the linear IC is near-zero but the effect only shows up
# once you're away from max pain; use signed-magnitude so the composite can
# still pick up direction while the |x|>1.5% regime effect dominates
df['max_pain_dist_abs'] = df['max_pain_dist_pct'].abs()

# VIX x basis interaction — significant in the OLS work, not yet in the composite
df['vix_x_basis'] = df['vix_close'] * df['basis']

# client_fut_net_pct — contrarian effect is concentrated in the "heavy long"
# bucket (20-50), not linear; a squared/deviation-from-neutral transform
# captures "how far from neutral" without assuming a sign-consistent slope
df['client_fut_net_dev'] = (df['client_fut_net_pct'] - df['client_fut_net_pct'].median()).abs()

In [54]:
import numpy as np
import pandas as pd
from scipy import stats

# factors = {'VIX': 'vix_close', 'PCR': 'pcr', ...} from cell 19 — use it to
# translate res['factor'] (display names) back to real column names.
disp_to_col = factors

STATE_IC_5D = {}
for state_id, state_label in state_names.items():
    sub = res[(res['state'] == state_label) & (res['horizon'] == '5D')]
    STATE_IC_5D[state_id] = {
        disp_to_col[fac]: ic for fac, ic in zip(sub['factor'], sub['ic'])
        if fac in disp_to_col
    }

full_sub = res[(res['state'] == 'Full') & (res['horizon'] == '5D')]
FULL_IC_5D = {
    disp_to_col[fac]: ic for fac, ic in zip(full_sub['factor'], full_sub['ic'])
    if fac in disp_to_col
}

N_STATES = best['n']   # single source of truth, set once right after model selection

# ── Cell 18: colors/labels — generate instead of hardcode ──
import matplotlib.cm as cm
_palette_base = [PAPER_BLUE, PAPER_RED, PAPER_ORANGE, PAPER_TEAL, PAPER_GOLD, PAPER_GRAY]
if N_STATES <= len(_palette_base):
    state_color_list = _palette_base[:N_STATES]
else:
    # fall back to a colormap if the model picked more states than we have named colors for
    state_color_list = [cm.tab10(i / N_STATES) for i in range(N_STATES)]

STATE_COLORS = {i: state_color_list[i] for i in range(N_STATES)}

# Generic labels — replace the hand-written 'S0: High Premium' etc, since
# those were written for a specific n=5 solution and won't describe an
# n=6+ model correctly. Rename descriptively AFTER inspecting m.means_ below.
STATE_LABELS = {i: f'S{i}' for i in range(N_STATES)}

STATE_MARKERS_LIST = ['o', '^', 's', 'D', 'v', 'P', 'X', '*', 'h', '8']
STATE_MARKERS = {i: STATE_MARKERS_LIST[i % len(STATE_MARKERS_LIST)] for i in range(N_STATES)}
state_colors = {0: PAPER_BLUE, 1: PAPER_RED, 2: PAPER_ORANGE, 3: PAPER_TEAL, 4: PAPER_PURPLE}
METHOD_STATIC = PAPER_GRAY
METHOD_REGIME = '#3a3a3a'

factor_cols = [
    'vix_close', 'pcr', 'max_pain_dist_pct', 'max_pain_dist_abs', 'basis',
    'cost_of_carry', 'fut_chng_oi_pct', 'fii_fut_net_pct', 'client_fut_net_pct',
    'client_fut_net_dev', 'fii_pe_net_pct', 'market_cap_z', 'vix_x_basis'
]

train = df[df['split'] == 'train'].copy().reset_index(drop=True)

for col in factor_cols:
    expanding_mean = train[col].expanding().mean()
    expanding_std  = train[col].expanding().std().replace(0, np.nan)
    train[f'z_{col}'] = (train[col] - expanding_mean) / expanding_std

train = train.dropna(subset=[f'z_{col}' for col in factor_cols] + ['fwd_ret_5d', 'hmm_state'])

# ── Vectorized static score (same weights for every row — pure matrix op) ──
static_cols    = list(FULL_IC_5D.keys())
static_weights = np.array([FULL_IC_5D[c] for c in static_cols])
Z_static = train[[f'z_{c}' for c in static_cols]].to_numpy()
train['score_static'] = (Z_static @ static_weights) / np.abs(static_weights).sum()

# ── Vectorized regime score — build per-row weight vectors once per state,
# then a single dot product per state group instead of per-row .apply ──
train['score_regime'] = np.nan
for state_id, weights in STATE_IC_5D.items():
    if not weights:
        continue
    mask = train['hmm_state'] == state_id
    if not mask.any():
        continue
    cols = list(weights.keys())
    w = np.array([weights[c] for c in cols])
    Z = train.loc[mask, [f'z_{c}' for c in cols]].to_numpy()
    train.loc[mask, 'score_regime'] = (Z @ w) / np.abs(w).sum()

def eval_composite(scores, returns, label):
    mask = scores.notna() & returns.notna()
    s, r = scores[mask], returns[mask]
    ic, p = stats.spearmanr(s, r)
    quintiles  = pd.qcut(s, 5, labels=False)
    q_means    = r.groupby(quintiles).mean()
    long_mask  = quintiles == 4
    short_mask = quintiles == 0
    ls_ret  = pd.concat([r[long_mask], -r[short_mask]])
    sharpe  = ls_ret.mean() / ls_ret.std() * np.sqrt(252/5)
    hit_rate = (r[long_mask] > 0).mean()
    print(f"\n{label}")
    print(f"  IC={ic:+.3f}  p={p:.4f}")
    print(f"  Q5-Q1 spread: {q_means.iloc[-1]-q_means.iloc[0]:+.2f}%")
    print(f"  L/S Annualized Sharpe: {sharpe:.2f}")
    print(f"  Top-quintile hit rate: {hit_rate:.1%}")
    print(f"  Quintile avg returns: {[f'{v:+.2f}%' for v in q_means]}")
    return {'ic': ic, 'p': p, 'spread': q_means.iloc[-1]-q_means.iloc[0],
            'sharpe': sharpe, 'hit_rate': hit_rate, 'q_means': q_means}

r_regime = eval_composite(train['score_regime'], train['fwd_ret_5d'], 'REGIME-CONDITIONED')
r_static = eval_composite(train['score_static'], train['fwd_ret_5d'], 'STATIC (full-sample)')

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Regime-Conditioned vs Static Composite Score — 5D Forward Return\n'
             'Train Set Only', fontsize=13.5, fontweight='bold', color='#1a1a1a')

q_labels = ['Q1\n(Short)', 'Q2', 'Q3', 'Q4', 'Q5\n(Long)']
x, w = np.arange(5), 0.35

ax = axes[0]
ax.bar(x - w/2, r_static['q_means'], w, color=METHOD_STATIC, alpha=0.85, label='Static')
ax.bar(x + w/2, r_regime['q_means'], w, color=METHOD_REGIME, alpha=0.85, label='Regime')
for i, (sv, rv) in enumerate(zip(r_static['q_means'], r_regime['q_means'])):
    ax.text(i - w/2, sv + np.sign(sv)*0.01, f'{sv:+.2f}', ha='center',
            va='bottom' if sv >= 0 else 'top', color='#1a1a1a', fontsize=7.5)
    ax.text(i + w/2, rv + np.sign(rv)*0.01, f'{rv:+.2f}', ha='center',
            va='bottom' if rv >= 0 else 'top', color='#1a1a1a', fontsize=7.5)
ax.axhline(0, color='#999999', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(q_labels, fontsize=9)
ax.set_ylabel('Avg 5D Return (%)', fontsize=9.5)
ax.set_title('(a) Quintile Returns', loc='left', fontsize=11)
ax.yaxis.grid(True, color='#e0e0e0', lw=0.6); ax.xaxis.grid(False)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=9)

ax = axes[1]
metrics = ['IC', 'Sharpe', 'Hit Rate']
s_vals  = [r_static['ic'], r_static['sharpe'], r_static['hit_rate']]
r_vals  = [r_regime['ic'],  r_regime['sharpe'],  r_regime['hit_rate']]
x2 = np.arange(3)
ax.bar(x2 - w/2, s_vals, w, color=METHOD_STATIC, alpha=0.85, label='Static')
ax.bar(x2 + w/2, r_vals, w, color=METHOD_REGIME, alpha=0.85, label='Regime')
for i, (sv, rv) in enumerate(zip(s_vals, r_vals)):
    ax.text(i - w/2, sv + 0.05, f'{sv:.2f}', ha='center', color='#1a1a1a', fontsize=7.5)
    ax.text(i + w/2, rv + 0.05, f'{rv:.2f}', ha='center', color='#1a1a1a', fontsize=7.5)
ax.axhline(0, color='#999999', lw=0.8)
ax.set_xticks(x2); ax.set_xticklabels(metrics, fontsize=9.5)
ax.set_title('(b) Summary Metrics', loc='left', fontsize=11)
ax.yaxis.grid(True, color='#e0e0e0', lw=0.6); ax.xaxis.grid(False)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=9)

ax = axes[2]
for state_id in range(N_STATES):
    mask = train['hmm_state'] == state_id
    ax.scatter(train.loc[mask, 'score_regime'], train.loc[mask, 'fwd_ret_5d'],
               color=STATE_COLORS[state_id], marker=STATE_MARKERS[state_id],
               alpha=0.8, s=28, edgecolors='none',
               label=f'S{state_id}')
ax.axhline(0, color='#999999', lw=0.8)
ax.axvline(0, color='#999999', lw=0.8, linestyle='--')
ax.set_xlabel('Regime Composite Score', fontsize=9.5)
ax.set_ylabel('5D Forward Return (%)', fontsize=9.5)
ax.set_title('(c) Score vs Return by State', loc='left', fontsize=11)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=8.5)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(RESEARCH_ROOT / 'plots/regime_composite_vs_static.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")


REGIME-CONDITIONED
  IC=+0.193  p=0.0000
  Q5-Q1 spread: +0.97%
  L/S Annualized Sharpe: 1.37
  Top-quintile hit rate: 67.1%
  Quintile avg returns: ['-0.15%', '-0.03%', '+0.13%', '+0.58%', '+0.82%']

STATIC (full-sample)
  IC=+0.120  p=0.0000
  Q5-Q1 spread: +0.53%
  L/S Annualized Sharpe: 0.71
  Top-quintile hit rate: 64.4%
  Quintile avg returns: ['+0.07%', '+0.27%', '+0.11%', '+0.30%', '+0.60%']
done


C:\Users\sriva\AppData\Local\Temp\ipykernel_11408\2623860411.py:166: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [55]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

n_boot = 2000
rng = np.random.default_rng(42)

def block_bootstrap_idx(n, block_size=10, rng=None):
    n_blocks = n // block_size + 1
    starts = rng.integers(0, n - block_size, size=n_blocks)
    idx = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
    return idx

def bootstrap_metrics(scores, returns, n_boot=2000, block_size=10):
    n = len(scores)
    ics, sharpes, hit_rates, spreads = [], [], [], []

    for _ in range(n_boot):
        idx = block_bootstrap_idx(n, block_size=block_size, rng=rng)   # <-- was rng.integers(0, n, size=n)
        s, r = scores.iloc[idx], returns.iloc[idx]

        ic, _ = spearmanr(s, r)

        quintiles = pd.qcut(s, 5, labels=False, duplicates='drop')
        q_means = r.groupby(quintiles).mean()
        if len(q_means) < 5:
            continue
        spread = q_means.iloc[-1] - q_means.iloc[0]

        long_mask  = quintiles == 4
        short_mask = quintiles == 0
        ls_ret = pd.concat([r[long_mask], -r[short_mask]])
        sharpe = ls_ret.mean() / ls_ret.std() * np.sqrt(252/5)
        hit    = (r[long_mask] > 0).mean()

        ics.append(ic)
        sharpes.append(sharpe)
        hit_rates.append(hit)
        spreads.append(spread)

    return {
        'ic':       np.array(ics),
        'sharpe':   np.array(sharpes),
        'hit_rate': np.array(hit_rates),
        'spread':   np.array(spreads),
    }

mask = train['score_regime'].notna() & train['score_static'].notna() & train['fwd_ret_5d'].notna()
t = train[mask].reset_index(drop=True)

print("Bootstrapping regime composite...")
boot_r = bootstrap_metrics(t['score_regime'], t['fwd_ret_5d'], n_boot)
print("Bootstrapping static composite...")
boot_s = bootstrap_metrics(t['score_static'], t['fwd_ret_5d'], n_boot)

diff_ic      = boot_r['ic']       - boot_s['ic']
diff_sharpe  = boot_r['sharpe']   - boot_s['sharpe']
diff_hit     = boot_r['hit_rate'] - boot_s['hit_rate']
diff_spread  = boot_r['spread']   - boot_s['spread']

def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

def p_gt_zero(arr):
    return (arr > 0).mean()

print("\n=== BOOTSTRAP RESULTS (n=2000, 95% CI) ===")
for name, b_r, b_s, diff in [
    ('IC',        boot_r['ic'],       boot_s['ic'],       diff_ic),
    ('Sharpe',    boot_r['sharpe'],   boot_s['sharpe'],   diff_sharpe),
    ('Hit Rate',  boot_r['hit_rate'], boot_s['hit_rate'], diff_hit),
    ('Q5-Q1 Spd', boot_r['spread'],  boot_s['spread'],   diff_spread),
]:
    lo_r, hi_r = ci95(b_r)
    lo_s, hi_s = ci95(b_s)
    lo_d, hi_d = ci95(diff)
    p = p_gt_zero(diff)
    print(f"\n{name}:")
    print(f"  Regime: {np.mean(b_r):.3f}  95% CI [{lo_r:.3f}, {hi_r:.3f}]")
    print(f"  Static: {np.mean(b_s):.3f}  95% CI [{lo_s:.3f}, {hi_s:.3f}]")
    print(f"  Diff:   {np.mean(diff):+.3f}  95% CI [{lo_d:.3f}, {hi_d:.3f}]  P(regime>static)={p:.3f}")

# --- Plot ---
# Reused from regime_composite_vs_static.png — same two methods being compared
METHOD_STATIC = PAPER_GRAY     # '#888888'
METHOD_REGIME = '#3a3a3a'      # dark neutral, distinct from the per-state palette

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Bootstrap CI — Regime vs Static Composite (n=2000)\n'
             '5D Forward Return | Train Set Only',
             fontsize=13.5, fontweight='bold', color='#1a1a1a')

panel_letters = ['(a)', '(b)', '(c)', '(d)']
pairs = [
    ('IC',         diff_ic,     boot_r['ic'],     boot_s['ic']),
    ('Sharpe',     diff_sharpe, boot_r['sharpe'], boot_s['sharpe']),
    ('Hit Rate',   diff_hit,    boot_r['hit_rate'], boot_s['hit_rate']),
    ('Q5-Q1 Spd',  diff_spread, boot_r['spread'], boot_s['spread']),
]

for letter, ax, (name, diff, br, bs) in zip(panel_letters, axes.flat, pairs):
    lo, hi = ci95(diff)
    p = p_gt_zero(diff)

    ax.hist(bs, bins=60, color=METHOD_STATIC, alpha=0.55, density=True, label='Static')
    ax.hist(br, bins=60, color=METHOD_REGIME, alpha=0.55, density=True, label='Regime')
    ax.set_ylabel('Density', fontsize=9)

    ax2 = ax.twinx()
    ax2.hist(diff, bins=60, color=PAPER_TEAL, alpha=0.35, density=True, label='Diff')
    ax2.axvline(0, color='#999999', lw=0.8, ls='--')
    ax2.axvline(lo, color=PAPER_TEAL, lw=0.8, ls=':')
    ax2.axvline(hi, color=PAPER_TEAL, lw=0.8, ls=':')
    ax2.axvline(np.mean(diff), color=PAPER_TEAL, lw=1.6)
    ax2.set_yticks([])
    ax2.grid(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    ci_str = f'[{lo:+.3f}, {hi:+.3f}]'
    significant = lo > 0 or hi < 0
    if lo > 0:
        sig_str, sig_color = '✓ significant (regime > static)', PAPER_TEAL
    elif hi < 0:
        sig_str, sig_color = '✓ significant (static > regime)', PAPER_RED
    else:
        sig_str, sig_color = '~ overlaps zero', '#777777'

    title_weight = 'bold' if significant else 'normal'
    title_color  = sig_color if significant else '#1a1a1a'

    ax.set_title(f'{letter} {name}  —  Diff 95% CI: {ci_str}\n{sig_str}',
                 loc='left', fontsize=10.5, color=title_color,
                 fontweight=title_weight, pad=10)

    ax.set_xlabel('Value', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=METHOD_STATIC, alpha=0.55),
        plt.Rectangle((0, 0), 1, 1, color=METHOD_REGIME, alpha=0.55),
        plt.Rectangle((0, 0), 1, 1, color=PAPER_TEAL, alpha=0.35),
    ]
    ax.legend(handles, ['Static', 'Regime', 'Diff'], loc='upper left',
              frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=8.5)

    ax.text(0.97, 0.05, f'P(R>S)={p:.3f}', transform=ax.transAxes,
            ha='right', fontsize=8.5, color=title_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor='#cccccc', linewidth=0.6))

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(RESEARCH_ROOT / 'plots/bootstrap_regime_vs_static.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")

Bootstrapping regime composite...
Bootstrapping static composite...

=== BOOTSTRAP RESULTS (n=2000, 95% CI) ===

IC:
  Regime: 0.192  95% CI [0.125, 0.261]
  Static: 0.118  95% CI [0.045, 0.188]
  Diff:   +0.074  95% CI [-0.021, 0.173]  P(regime>static)=0.934

Sharpe:
  Regime: 1.438  95% CI [0.427, 2.427]
  Static: 0.778  95% CI [-0.256, 1.873]
  Diff:   +0.660  95% CI [-0.744, 2.166]  P(regime>static)=0.799

Hit Rate:
  Regime: 0.675  95% CI [0.613, 0.733]
  Static: 0.646  95% CI [0.580, 0.710]
  Diff:   +0.029  95% CI [-0.058, 0.121]  P(regime>static)=0.727

Q5-Q1 Spd:
  Regime: 0.976  95% CI [0.358, 1.568]
  Static: 0.536  95% CI [-0.252, 1.181]
  Diff:   +0.440  95% CI [-0.397, 1.387]  P(regime>static)=0.825
done


C:\Users\sriva\AppData\Local\Temp\ipykernel_11408\672764837.py:156: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [56]:
df_s1 = df[df['hmm_state'] == 1].copy()

events = {
    '2019_stress': ('2019-07-15', '2019-10-15'),
    'covid':        ('2020-02-01', '2020-08-01'),
}

df_s1['event'] = 'other'
for name, (start, end) in events.items():
    mask = (df_s1['trade_date'] >= start) & (df_s1['trade_date'] <= end)
    df_s1.loc[mask, 'event'] = name

print(df_s1['event'].value_counts())

# leave-one-event-out ICs
for excl_name in events:
    sub = df_s1[df_s1['event'] != excl_name]
    ic, p = spearman_ic(sub['vix_close'], sub['fwd_ret_20d'])
    print(f"S1 excluding {excl_name:12s}: n={len(sub)}  VIX IC={ic:+.3f}  p={p:.3f}")

# both excluded together
sub_both = df_s1[df_s1['event'] == 'other']
ic, p = spearman_ic(sub_both['vix_close'], sub_both['fwd_ret_20d'])
print(f"S1 excluding both:        n={len(sub_both)}  VIX IC={ic:+.3f}  p={p:.3f}")

event
other    373
covid     22
Name: count, dtype: int64
S1 excluding 2019_stress : n=395  VIX IC=+0.211  p=0.000
S1 excluding covid       : n=373  VIX IC=+0.240  p=0.000
S1 excluding both:        n=373  VIX IC=+0.240  p=0.000


In [57]:
con.close()